## 1. Importação de Bibliotecas

In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

## 2. Configuração do Ambiente

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

BASE_DIR = Path(".")
FIGURES_DIR = BASE_DIR / "figures"
TABLES_DIR = BASE_DIR / "tables"

FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

N_CLUSTERS = 4
RANDOM_STATE = 42

## 3. Funções Auxiliares

In [ ]:
def load_excel(filename: str) -> pd.DataFrame:
    path = BASE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_excel(path)


def save_figure(fig: plt.Figure, filename: str) -> None:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")

## 4. Carregamento dos Dados

In [ ]:
df_ativos = load_excel("Estudantes_ativos_EP_2025.xlsx")
df_inativos = load_excel("Estudantes_inativos_EP_2025.xlsx")
df_concat = load_excel("Estudantes_EP_2025_concat.xlsx")

df_inativos = df_inativos[df_inativos['Ano_Ingresso'] >= 2012]
df_concat = df_concat[df_concat['Ano_Ingresso'] >= 2012]

## 5. Funções de Seleção de Features

In [ ]:
def preparar_subconjunto_numerico(df, features):
    presentes = [f for f in features if f in df.columns]
    if not presentes:
        return None
    sub = df[presentes].apply(pd.to_numeric, errors="coerce")
    return sub


def avaliar_features_cluster(df, features, nome_base="Base de referência"):
    sub = preparar_subconjunto_numerico(df, features)
    if sub is None:
        return None, None, None
    
    missing = sub.isnull().sum() / len(sub)
    corr = sub.corr().abs()
    
    return sub, corr, missing


def selecionar_features(sub_df, corr, missing, max_missing=0.4, corr_threshold=0.9):
    if sub_df is None:
        return [], list(missing.index) if missing is not None else []
    
    validas = missing[missing <= max_missing].index.tolist()
    
    if len(validas) < 2:
        return validas, [f for f in missing.index if f not in validas]
    
    corr_subset = corr.loc[validas, validas]
    
    to_drop = set()
    for i in range(len(corr_subset.columns)):
        for j in range(i + 1, len(corr_subset.columns)):
            if corr_subset.iloc[i, j] > corr_threshold:
                to_drop.add(corr_subset.columns[j])
    
    selecionadas = [f for f in validas if f not in to_drop]
    descartadas = [f for f in missing.index if f not in selecionadas]
    
    return selecionadas, descartadas

## 6. Seleção de Features para Clustering

In [ ]:
features_cluster = [
    "IRA",
    "Perfil",
    "Media_Total",
    "Ponderada_Total",
    "Porcentagem_Concluido_SIGA",
    "Porcentagem_Inscrito",
    "Porcentagem_Aprovado",
    "Porcentagem_Reprovado",
    "Numero_Horas_Inscritas_Resultado",
    "Horas_Aprovadas",
    "Nota_SISU",
]

sub_concat, corr_concat, missing_concat = avaliar_features_cluster(
    df_ativos, features_cluster, nome_base="df_ativos"
)

features_usadas, features_descartadas = selecionar_features(
    sub_concat, corr_concat, missing_concat, max_missing=0.4, corr_threshold=0.9
)

display(Markdown("### Features para clusterização"))

display(
    Markdown(
        f"**Features utilizadas:** {', '.join(features_usadas) if features_usadas else 'Nenhuma selecionada.'}"
    )
)

if features_descartadas:
    display(Markdown(f"**Features descartadas:** {', '.join(features_descartadas)}"))

## 7. Função Principal de K-Means + PCA

In [ ]:
def rodar_kmeans_pca(
    df,
    feature_cols,
    n_clusters=N_CLUSTERS,
    nome_cenario="",
    random_state=RANDOM_STATE,
    prefixo_arquivo=None,
):
    if not feature_cols:
        return None
    
    feature_cols = [f for f in feature_cols if f in df.columns]
    df_clean = df[feature_cols].dropna()
    
    if len(df_clean) == 0:
        return None
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clean)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    clusters = kmeans.fit_predict(X_scaled)
    
    silhouette = silhouette_score(X_scaled, clusters)
    
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    df_pca["Cluster"] = clusters
    
    fig, ax = plt.subplots(figsize=(8, 6))
    for cluster_id in range(n_clusters):
        subset = df_pca[df_pca["Cluster"] == cluster_id]
        ax.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {cluster_id}", alpha=0.6)
    
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})")
    ax.legend()
    ax.set_title(f"{nome_cenario} - Silhouette: {silhouette:.3f}")
    
    if prefixo_arquivo:
        save_figure(fig, f"{prefixo_arquivo}_pca.png")
    plt.show()
    
    df_clean_copy = df_clean.copy()
    df_clean_copy["Cluster"] = clusters
    medias = df_clean_copy.groupby("Cluster")[feature_cols].mean()
    
    if prefixo_arquivo:
        medias.to_excel(TABLES_DIR / f"{prefixo_arquivo}_medias_cluster.xlsx")
    
    display(medias)
    
    loadings = pd.DataFrame(
        pca.components_.T,
        columns=["PC1", "PC2"],
        index=feature_cols
    )
    
    if prefixo_arquivo:
        loadings.to_excel(TABLES_DIR / f"{prefixo_arquivo}_loadings.xlsx")
    
    display(loadings)
    
    if "IRA" in df.columns and "Porcentagem_Concluido_SIGA" in df.columns:
        df_scatter = df.copy()
        df_scatter = df_scatter.loc[df_clean.index]
        df_scatter["Cluster"] = clusters
        
        fig, ax = plt.subplots(figsize=(8, 6))
        for cluster_id in range(n_clusters):
            subset = df_scatter[df_scatter["Cluster"] == cluster_id]
            ax.scatter(
                subset["Porcentagem_Concluido_SIGA"],
                subset["IRA"],
                label=f"Cluster {cluster_id}",
                alpha=0.6
            )
        
        ax.set_xlabel("Porcentagem concluída (%)")
        ax.set_ylabel("IRA")
        ax.legend()
        
        if prefixo_arquivo:
            save_figure(fig, f"{prefixo_arquivo}_ira_concluido.png")
        plt.show()
    
    return {
        "kmeans": kmeans,
        "clusters": clusters,
        "silhouette": silhouette,
        "pca": pca,
        "X_pca": X_pca,
        "X_scaled": X_scaled,
        "df_clean": df_clean,
        "medias": medias,
        "loadings": loadings,
    }

## 8. Clustering - Ativos (IRA qualquer)

In [ ]:
df_ativos_sem_filtro = df_ativos.copy()
df_ativos_sem_filtro["IRA"] = pd.to_numeric(df_ativos_sem_filtro["IRA"], errors="coerce")

resultado_ativos_sem_filtro = rodar_kmeans_pca(
    df_ativos_sem_filtro,
    features_usadas,
    n_clusters=N_CLUSTERS,
    nome_cenario="Ativos (IRA qualquer)",
    prefixo_arquivo="ativos_ira_qualquer",
)

## 9. Clustering - Ativos (IRA > 0)

In [ ]:
df_ativos_filtrado = df_ativos.copy()
df_ativos_filtrado["IRA"] = pd.to_numeric(df_ativos_filtrado["IRA"], errors="coerce")
df_ativos_filtrado = df_ativos_filtrado[df_ativos_filtrado["IRA"] > 0]

resultado_ativos_filtrado = rodar_kmeans_pca(
    df_ativos_filtrado,
    features_usadas,
    n_clusters=N_CLUSTERS,
    nome_cenario="Ativos com IRA > 0",
    prefixo_arquivo="ativos_ira_maior_zero",
)

## 10. Clustering - Concat (Ativos + Inativos, IRA > 0)

In [ ]:
df_concat_filtrado = df_concat.copy()
df_concat_filtrado["IRA"] = pd.to_numeric(df_concat_filtrado["IRA"], errors="coerce")
df_concat_filtrado = df_concat_filtrado[df_concat_filtrado["IRA"] > 0]

resultado_concat_filtrado = rodar_kmeans_pca(
    df_concat_filtrado,
    features_usadas,
    n_clusters=N_CLUSTERS,
    nome_cenario="Concat (Ativos + Inativos, IRA > 0)",
    prefixo_arquivo="concat_ira_maior_zero",
)

## 11. Distribuição de Status por Cluster (Ativos Filtrado)

In [ ]:
if resultado_ativos_filtrado is not None:
    df_status = df_ativos_filtrado.copy()
    df_status = df_status.loc[resultado_ativos_filtrado["df_clean"].index]
    df_status["Cluster"] = resultado_ativos_filtrado["clusters"]
    
    dist_status = (
        df_status.groupby(["Cluster", "Status"])
        .size()
        .reset_index(name="Count")
    )
    
    total_cluster = dist_status.groupby("Cluster")["Count"].transform("sum")
    dist_status["Proporcao"] = (dist_status["Count"] / total_cluster) * 100
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    clusters = sorted(dist_status["Cluster"].unique())
    status_list = dist_status["Status"].unique()
    
    x = np.arange(len(clusters))
    width = 0.8 / len(status_list)
    
    for i, status in enumerate(status_list):
        subset = dist_status[dist_status["Status"] == status]
        valores = [subset[subset["Cluster"] == c]["Proporcao"].sum() for c in clusters]
        ax.bar(x + i * width, valores, width, label=status)
    
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Proporção (%)")
    ax.set_xticks(x + width * len(status_list) / 2)
    ax.set_xticklabels([f"Cluster {c}" for c in clusters])
    ax.legend()
    
    save_figure(fig, "ativos_filtrado_status_por_cluster.png")
    plt.show()

## 12. Distribuição de Status por Cluster (Concat Filtrado)

In [ ]:
if resultado_concat_filtrado is not None:
    df_status = df_concat_filtrado.copy()
    df_status = df_status.loc[resultado_concat_filtrado["df_clean"].index]
    df_status["Cluster"] = resultado_concat_filtrado["clusters"]
    
    dist_status = (
        df_status.groupby(["Cluster", "Status"])
        .size()
        .reset_index(name="Count")
    )
    
    total_cluster = dist_status.groupby("Cluster")["Count"].transform("sum")
    dist_status["Proporcao"] = (dist_status["Count"] / total_cluster) * 100
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    clusters = sorted(dist_status["Cluster"].unique())
    status_list = dist_status["Status"].unique()
    
    x = np.arange(len(clusters))
    width = 0.8 / len(status_list)
    
    for i, status in enumerate(status_list):
        subset = dist_status[dist_status["Status"] == status]
        valores = [subset[subset["Cluster"] == c]["Proporcao"].sum() for c in clusters]
        ax.bar(x + i * width, valores, width, label=status)
    
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Proporção (%)")
    ax.set_xticks(x + width * len(status_list) / 2)
    ax.set_xticklabels([f"Cluster {c}" for c in clusters])
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    
    save_figure(fig, "concat_filtrado_status_por_cluster.png")
    plt.show()